# Повторная госпитализация: Spark и AutoML
Вариант 14. Данные UCI Diabetes 130-US hospitals (1999–2008): 101766 записей. Методичка разрешает исходный набор без искусственного увеличения. Цель — повторная госпитализация ранее 30 дней (`<30`), остальные исходы относятся к классу 0.
Перед разделением оставляем первую зарегистрированную запись пациента по encounter_id, чтобы один пациент не пересекал train, CV и test. Признаки доступны к моменту выписки. Это учебный эксперимент, а не клиническая система.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.readmission import prepare, train, reload_check
data, counts = prepare()

root
 |-- encounter_id: integer (nullable = true)
 |-- patient_nbr: integer (nullable = true)
 |-- race: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: string (nullable = true)
 |-- weight: string (nullable = true)
 |-- admission_type_id: integer (nullable = true)
 |-- discharge_disposition_id: integer (nullable = true)
 |-- admission_source_id: integer (nullable = true)
 |-- time_in_hospital: double (nullable = true)
 |-- payer_code: string (nullable = true)
 |-- medical_specialty: string (nullable = true)
 |-- num_lab_procedures: double (nullable = true)
 |-- num_procedures: double (nullable = true)
 |-- num_medications: double (nullable = true)
 |-- number_outpatient: double (nullable = true)
 |-- number_emergency: double (nullable = true)
 |-- number_inpatient: double (nullable = true)
 |-- diag_1: string (nullable = true)
 |-- diag_2: string (nullable = true)
 |-- diag_3: string (nullable = true)
 |-- number_diagnoses: double (nullable = true)
 |-- max_gl

+------------+-----------+---------+------+-------+---------+-----------------+------------------------+-------------------+----------------+----------+----------------------+------------------+--------------+---------------+-----------------+----------------+----------------+------+------+------+----------------+-------------+---------+---------+-----------+-----------+--------------+-----------+-------------+---------+---------+-----------+------------+-------------+--------+--------+------------+----------+-------+-----------+-------+-------------------+-------------------+------------------------+-----------------------+----------------------+------+-----------+----------+------------------+-------------------+-----------------+-----------------+
|encounter_id|patient_nbr|race     |gender|age    |weight   |admission_type_id|discharge_disposition_id|admission_source_id|time_in_hospital|payer_code|medical_specialty     |num_lab_procedures|num_procedures|num_medications|number_outpati

{'raw_rows': 101766, 'deduplicated_encounters': 101766, 'unique_patients': 71518}


## Выбор модели только на обучающих данных
Сначала отделяем стратифицированный test 20%, затем выбираем 18000 обучающих пациентов. PyCaret обучает заполнение пропусков, кодирование и нормализацию внутри трёхфолдовой стратифицированной CV. Два поиска имеют бюджет по две минуты и по три модели. В первом поиске логистическая регрессия служит простой базовой моделью. Победителя выбираем по CV ROC-AUC, test используется только для итоговых метрик.
В Spark создаются total_prior_visits, medications_per_day и labs_per_day. Идентификаторы и исходная целевая колонка не входят в признаки. Финальное обучение PyCaret использует только обучающую подвыборку.

In [2]:
result = train(data, counts)
import pandas as pd
display(pd.DataFrame(result["runs"]))

2026/11/21 23:32:48 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/11/21 23:32:48 INFO mlflow.store.db.utils: Updating database tables


INFO  [alembic.runtime.migration] Context impl SQLiteImpl.


INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


INFO  [alembic.runtime.migration] Running upgrade  -> 451aebb31d03, add metric step


INFO  [alembic.runtime.migration] Running upgrade 451aebb31d03 -> 90e64c465722, migrate user column to tags


... (34 outputs left out)

Таблица кросс-валидации: boosting-discriminant
                            Model  Accuracy     AUC  Recall   Prec.      F1   Kappa     MCC  TT (Sec)
lda  Linear Discriminant Analysis    0.9054  0.6557  0.0640  0.3204  0.1055  0.0786  0.1084    0.2567
gbc  Gradient Boosting Classifier    0.9119  0.6546  0.0234  0.4843  0.0447  0.0370  0.0916    1.0267
ada          Ada Boost Classifier    0.9112  0.6470  0.0180  0.4204  0.0345  0.0271  0.0710    0.5200


2026/11/21 23:33:14 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


{
  "counts": {
    "raw_rows": 101766,
    "deduplicated_encounters": 101766,
    "unique_patients": 71518,
    "training_pool": 57214,
    "automl_rows": 18000,
    "holdout_rows": 14304,
    "training_positives": 1584,
    "holdout_positives": 1259,
    "patient_overlap": 0
  },
  "versions": {
    "pycaret": "3.3.2",
    "pyspark": "3.5.9",
    "scikit-learn": "1.4.2",
    "pandas": "2.1.4",
    "mlflow": "2.22.5"
  },
  "runs": [
    {
      "run": "linear-trees",
      "run_id": "280d68936e5649e9bca8fc8d639c2726",
      "selected": "LogisticRegression",
      "cv_auc": 0.6531,
      "roc_auc": 0.6427090072215959,
      "accuracy": 0.9117030201342282,
      "f1": 0.01096319498825372,
      "pr_auc": 0.16753721124551718
    },
    {
      "run": "boosting-discriminant",
      "run_id": "ea5310ccd8ed4915a3935edc75261440",
      "selected": "LinearDiscriminantAnalysis",
      "cv_auc": 0.6557,
      "roc_auc": 0.6455538672725407,
      "accuracy": 0.908417225950783,
      "f1": 0.085

,run,run_id,selected,cv_auc,roc_auc,accuracy,f1,pr_auc
0,linear-trees,280d68936e5649e9bca8fc8d639c2726,LogisticRegression,0.6531,0.642709,0.911703,0.010963,0.167537
1,boosting-discriminant,ea5310ccd8ed4915a3935edc75261440,LinearDiscriminantAnalysis,0.6557,0.645554,0.908417,0.085196,0.172922


## Сохранение и воспроизводимость
Модель сохраняется вместе с предобработкой. Данные и модели отслеживает DVC. MLflow локально хранит параметры, версии, размеры выборок, таблицы сравнения, метрики и модели. ROC-AUC и average precision оценивают ранжирование, F1 и accuracy — ответы при стандартном пороге 0.5, который не подбирался по test.

In [3]:
probabilities = reload_check(result)
display(probabilities)

Загружен полный конвейер предобработки: (10, 2)


array([[0.95847118, 0.04152882],
       [0.9671373 , 0.0328627 ],
       [0.87538357, 0.12461643],
       [0.95565477, 0.04434523],
       [0.9680699 , 0.0319301 ],
       [0.9394494 , 0.0605506 ],
       [0.93475577, 0.06524423],
       [0.94782258, 0.05217742],
       [0.78547396, 0.21452604],
       [0.87441516, 0.12558484]])

## Выводы и ограничения
Для 101766 записей pandas был бы проще, но Spark демонстрирует очистку, типизацию и создание признаков с возможностью перехода к большим данным. AutoML сокращает ручной код сравнения моделей, но не заменяет постановку задачи и контроль утечек. Малые различия CV не доказывают устойчивого преимущества. Низкий F1 при пороге 0.5 ограничивает обнаружение редкого исхода; для медицинского применения нужны временная и внешняя проверки, оценка калибровки и клинической пользы.